# 📊 Jobs Data Analysis Notebook

**Project Overview:**  
This notebook analyzes the **cleaned jobs dataset** to extract insights. 

**Dataset:** Cleaned jobs CSV created from the previous notebook.

## 1️⃣ Imports

We start by importing the necessary library for analysis:

- **pandas**: For data manipulation and analysis

In [1]:
import pandas as pd

## 2️⃣ Load and Inspect Data

We begin by loading the **cleaned jobs CSV** created in the previous notebook.  

- Check the first few rows to confirm the data loaded correctly.  
- Inspect the dataset to see column names, data types, and overall structure.  
- Ensure there are no unexpected missing values or issues before analysis.

In [4]:
# Load cleaned CSV 
df = pd.read_csv("../data/processed/cleaned_jobs.csv") 
# Quick check 
df.head()

,title,company,date_posted,skills_required,job_desc,link,city,state
0,Developer,CIBC India,2026-01-20 08:35:48+00:00,"['ai', 'python']",What Youll Be Doing (position summary): As a F...,https://www.adzuna.in/land/ad/5591948786?se=Ti...,Hyderabad,Telangana
1,Developer,Birlasoft,2026-02-17 09:43:06+00:00,"['aws', 'ai', 'python']",Job Description (Developer) Generative AI Deve...,https://www.adzuna.in/land/ad/5632728089?se=Ti...,Hyderabad,Telangana
2,Developer,CIBC India,2026-02-18 09:39:25+00:00,['ai'],What Youll Be Doing (position summary): Design...,https://www.adzuna.in/land/ad/5634049986?se=Ti...,Hyderabad,Telangana
3,Developer,Terra Technology Circle Consulting Private Lim...,2026-02-21 02:58:09+00:00,['ai'],The profile open is for the role of a develope...,https://www.adzuna.in/land/ad/5638899970?se=Ti...,Delhi,NaN
4,Brand Development & Channel Development Manager,Lubz Corporation,2026-02-17 09:37:06+00:00,['git'],Industry: Automobile Department: Sales & Marke...,https://www.adzuna.in/land/ad/5632720468?se=Ti...,Mumbai,Maharashtra


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   title            1000 non-null   str  
 1   company          1000 non-null   str  
 2   date_posted      1000 non-null   str  
 3   skills_required  1000 non-null   str  
 4   job_desc         1000 non-null   str  
 5   link             1000 non-null   str  
 6   city             823 non-null    str  
 7   state            807 non-null    str  
dtypes: str(8)
memory usage: 62.6 KB


In [6]:
import sys
import os

# Instead of __file__, use the current working directory
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from config.settings import Settings
skills = Settings().SKILL_KEYWORDS

## 3️⃣ Initialize Skill Counts

Before analyzing job descriptions, we create a dictionary to **track how often each skill appears**.  

- Each skill from our predefined `skills` list is a key in the dictionary.  
- The value for each key is initialized to `0`.  
- As we process job descriptions, we will increment these counts.

In [9]:
import ast

# Convert 'skills_required' string to actual list and lowercase all skills
df['skills_required_list'] = df['skills_required'].apply(lambda x: [s.lower() for s in ast.literal_eval(x)])

# Lowercase the skill keywords too
skill_keywords = [s.lower() for s in skills]

# Flatten all skills and keep only those in skill_keywords
all_skills = [skill for sublist in df['skills_required_list'] for skill in sublist if skill in skill_keywords]

# Count frequency
skill_counts = pd.Series(all_skills).value_counts().reset_index()
skill_counts.columns = ['Skill', 'Count']

# Resulting skill_df
skill_df = skill_counts

In [11]:
print(skill_df)

               Skill  Count
0                 ai    533
1                sql    123
2                git    110
3              react     95
4             python     83
5         javascript     80
6                aws     43
7               node     41
8              azure     39
9         typescript     30
10            docker     15
11        kubernetes     12
12           mongodb     11
13               gcp     10
14        postgresql     10
15            django      9
16             flask      8
17           fastapi      7
18  data engineering      5
19  machine learning      5
20            pandas      2


In [12]:
# Drop rows where 'state' is null
df_location = df.dropna(subset=['state'])

# Lowercase the states for consistency
df_location['state'] = df_location['state'].str.lower().str.strip()

# Count number of jobs per state
location_counts = df_location['state'].value_counts().reset_index()
location_counts.columns = ['State', 'Count']

# Resulting location_df
location_df = location_counts

In [13]:
location_df

,State,Count
0,maharashtra,318
1,telangana,268
2,karnataka,122
3,tamil nadu,36
4,ghaziabad,19
5,gujarat,16
6,west bengal,6
7,delhi,6
8,rajasthan,3
9,madhya pradesh,2


In [14]:
# Lowercase and strip company names for consistency
df['company_clean'] = df['company'].str.lower().str.strip()

# Count number of job postings per company
company_counts = df['company_clean'].value_counts().reset_index()
company_counts.columns = ['Company', 'Count']

# Resulting company_df
company_df = company_counts

# Save as CSV if needed
print(company_df)

                                               Company  Count
0                                   persistent systems     60
1                            tata consultancy services     53
2                                          ltimindtree     25
3                                               luxoft     19
4                           brace infotech private ltd     11
..                                                 ...    ...
565                                             kasplo      1
566                                          methodhub      1
567                  pluswealth capital management llp      1
568  agile technology solutions - your technology p...      1
569                        rivi consulting group l.l.c      1

[570 rows x 2 columns]


In [16]:
# Convert date_posted to datetime
df['date_posted_dt'] = pd.to_datetime(df['date_posted'])

# Extract month name
df['month'] = df['date_posted_dt'].dt.month_name()

# Count number of jobs per month (across all years)
jobs_per_month_counts = df['month'].value_counts().reindex([
    'January', 'February', 'March', 'April', 'May', 'June',
    'July', 'August', 'September', 'October', 'November', 'December'
]).reset_index()
jobs_per_month_counts.columns = ['Month', 'Count']

# Resulting DataFrame
jobs_per_month_df = jobs_per_month_counts
print(jobs_per_month_df)

        Month  Count
0     January   79.0
1    February  887.0
2       March    NaN
3       April    2.0
4         May    3.0
5        June    5.0
6        July    1.0
7      August    NaN
8   September    2.0
9     October    2.0
10   November   11.0
11   December    8.0


In [17]:
import pandas as pd
import ast

# Convert date_posted to datetime
df['date_posted_dt'] = pd.to_datetime(df['date_posted'])

# Sort by date (oldest → newest)
df = df.sort_values('date_posted_dt')

# Convert skills_required to list and lowercase
df['skills_required_list'] = df['skills_required'].apply(lambda x: [s.lower() for s in ast.literal_eval(x)])

# Lowercase skill keywords
skill_keywords_lower = [s.lower() for s in skills]

# Explode skills to one per row
df_exploded = df.explode('skills_required_list')

# Keep only relevant skills
df_exploded = df_exploded[df_exploded['skills_required_list'].isin(skill_keywords_lower)]

# Set date_posted as index for resampling
df_exploded.set_index('date_posted_dt', inplace=True)

# Group by week and skill, then count jobs
weekly_trends = df_exploded.groupby([pd.Grouper(freq='W'), 'skills_required_list']).size().reset_index(name='Count')

# Pivot so each skill is a column
weekly_trends_df = weekly_trends.pivot(index='date_posted_dt', columns='skills_required_list', values='Count').fillna(0)

# Sort by date (oldest → newest)
weekly_trends_df = weekly_trends_df.sort_index()

# Save as CSV
weekly_trends_df

skills_required_list,ai,aws,azure,data engineering,django,docker,fastapi,flask,gcp,git,...,kubernetes,machine learning,mongodb,node,pandas,postgresql,python,react,sql,typescript
date_posted_dt,,,,,,,,,,,,,,,,,,,,,
2025-04-27 00:00:00+00:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2025-05-18 00:00:00+00:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2025-05-25 00:00:00+00:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2025-06-08 00:00:00+00:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2025-06-22 00:00:00+00:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2025-09-07 00:00:00+00:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2025-09-28 00:00:00+00:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2025-10-19 00:00:00+00:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
2025-10-26 00:00:00+00:00,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# END OF NOTEBOOK 